# Step 4: Mapper Evaluation

## Schema Mapping Engine Performance Analysis

This notebook evaluates the SchemaMapEngine's performance by:
1. Loading the generated mapping output
2. Comparing mappings to the gold standard (curated metadata)
3. Calculating accuracy metrics (precision, recall, F1)
4. Analyzing confidence score distribution
5. Identifying failure cases and patterns

**Input files:**
- `../metadata_samples/curated_meta.csv` (Gold standard - 153 columns)
- `../data/schema_mapping_eval/new_meta.csv` (Mapper output - mappings with scores)

**Output:** Evaluation metrics, accuracy statistics, and recommendations

## 1. Setup & Load Data

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

print("Libraries imported")

Libraries imported


In [2]:
# Define paths
base_dir = Path("..")
metadata_samples_dir = base_dir / "metadata_samples"
eval_output_dir = base_dir / "data" / "schema_mapping_eval"

curated_path = metadata_samples_dir / "curated_meta.csv"
new_meta_path = metadata_samples_dir / "new_meta.csv"
mapper_output_path = eval_output_dir / "new_meta.csv"

print(f"Looking for files in:")
print(f"  Curated: {curated_path}")
print(f"  Mapper output: {mapper_output_path}\n")

# Load curated metadata (gold standard)
try:
    df_curated = pd.read_csv(curated_path)
    print(f"Loaded curated_meta.csv: {df_curated.shape}")
except FileNotFoundError:
    print(f"NOT FOUND: {curated_path}")
    df_curated = None

# Load mapper output
try:
    df_mapper_output = pd.read_csv(mapper_output_path)
    print(f"Loaded mapper output: {df_mapper_output.shape}")
except FileNotFoundError:
    print(f"NOT FOUND: {mapper_output_path}")
    print(f"   Have you run test_mapper.py yet?")
    print(f"   Command: cd ../ && python test_mapper.py")
    df_mapper_output = None

Looking for files in:
  Curated: ..\metadata_samples\curated_meta.csv
  Mapper output: ..\data\schema_mapping_eval\new_meta.csv

Loaded curated_meta.csv: (21881, 37)
Loaded mapper output: (10, 9)


In [3]:
# Display sample mapper output
if df_mapper_output is not None:
    print("\nMapper Output Sample (first 10 rows):\n")
    print(df_mapper_output.head(10))
    print(f"\nColumns in output: {df_mapper_output.columns.tolist()}")


Mapper Output Sample (first 10 rows):

           query  stage    method             match1  match1_score           match2  match2_score             match3  match3_score
0       column_1      1     exact          age_years          0.95     age_in_years          0.92                age          0.88
1       column_2      2     fuzzy     biological_sex          0.87              sex          0.85             gender          0.80
2       column_3      3  semantic  specimen_location          0.78    sample_source          0.75           location          0.70
3       column_4      1     exact   disease_category          0.92     disease_name          0.88            disease          0.85
4       column_5      4       llm    body_mass_index          0.65              bmi          0.62       weight_index          0.58
5   sample_col_6      2     fuzzy          age_years          0.81   biological_sex          0.79  specimen_location          0.75
6   sample_col_7      3  semantic   disease

## 2. Create Gold Standard Mapping

In [4]:
if df_curated is not None:
    print("="*80)
    print("CREATING GOLD STANDARD MAPPING")
    print("="*80)
    
    # The curated metadata columns are the "correct" mappings
    gold_standard = set(df_curated.columns)
    
    print(f"\nGold standard: {len(gold_standard)} standardized fields")
    print(f"\nFirst 20 standard fields:")
    for i, col in enumerate(sorted(gold_standard)[:20], 1):
        print(f"  {i:2d}. {col}")

CREATING GOLD STANDARD MAPPING

Gold standard: 37 standardized fields

First 20 standard fields:
   1. age_group
   2. age_group_ontology_term_id
   3. age_max
   4. age_min
   5. age_years
   6. antibiotics_current_use
   7. biomarker
   8. body_site
   9. body_site_ontology_term_id
  10. control
  11. control_ontology_term_id
  12. country
  13. country_ontology_term_id
  14. dietary_restriction
  15. disease
  16. disease_ontology_term_id
  17. feces_phenotype_metric
  18. feces_phenotype_metric_ontology_term_id
  19. feces_phenotype_value
  20. fmt_id


## 3. Analyze Mapper Output

In [5]:
if df_mapper_output is not None:
    print("\n" + "="*80)
    print("MAPPER OUTPUT ANALYSIS")
    print("="*80)
    
    print(f"\nOutput Statistics:")
    print(f"  • Total mappings: {len(df_mapper_output)}")
    print(f"  • Unique query columns: {df_mapper_output['query'].nunique()}")
    print(f"  • Unique match1 targets: {df_mapper_output['match1'].nunique()}")
    
    # Stage distribution
    if 'stage' in df_mapper_output.columns:
        print(f"\nMatching Stages:")
        stage_dist = df_mapper_output['stage'].value_counts().sort_index()
        for stage, count in stage_dist.items():
            pct = (count / len(df_mapper_output)) * 100
            print(f"  Stage {stage}: {count:4d} ({pct:5.1f}%)")
    
    # Method distribution
    if 'method' in df_mapper_output.columns:
        print(f"\nMatching Methods:")
        method_dist = df_mapper_output['method'].value_counts()
        for method, count in method_dist.head(10).items():
            pct = (count / len(df_mapper_output)) * 100
            print(f"  {method:20s}: {count:4d} ({pct:5.1f}%)")


MAPPER OUTPUT ANALYSIS

Output Statistics:
  • Total mappings: 10
  • Unique query columns: 10
  • Unique match1 targets: 5

Matching Stages:
  Stage 1:    3 ( 30.0%)
  Stage 2:    3 ( 30.0%)
  Stage 3:    3 ( 30.0%)
  Stage 4:    1 ( 10.0%)

Matching Methods:
  exact               :    3 ( 30.0%)
  fuzzy               :    3 ( 30.0%)
  semantic            :    3 ( 30.0%)
  llm                 :    1 ( 10.0%)


In [6]:
# Confidence score analysis
if df_mapper_output is not None and 'match1_score' in df_mapper_output.columns:
    print(f"\nConfidence Score (match1_score) Statistics:")
    
    scores = pd.to_numeric(df_mapper_output['match1_score'], errors='coerce')
    
    print(f"  Mean score    : {scores.mean():.4f}")
    print(f"  Median score  : {scores.median():.4f}")
    print(f"  Std dev       : {scores.std():.4f}")
    print(f"  Min score     : {scores.min():.4f}")
    print(f"  Max score     : {scores.max():.4f}")
    print(f"  25th percentile: {scores.quantile(0.25):.4f}")
    print(f"  75th percentile: {scores.quantile(0.75):.4f}")
    
    # Score distribution by confidence level
    print(f"\nScore Distribution by Confidence Level:")
    high_confidence = (scores >= 0.90).sum()     # >0.90
    good_confidence = (scores >= 0.80).sum()     # 0.80-0.90
    moderate_confidence = (scores >= 0.70).sum() # 0.70-0.80
    low_confidence = (scores >= 0.60).sum()      # 0.60-0.70
    very_low = (scores < 0.60).sum()             # <0.60
    
    total = len(scores)
    print(f"  • 0.90-1.00 (Excellent)  : {high_confidence:4d} ({high_confidence/total*100:5.1f}%)")
    print(f"  • 0.80-0.89 (Good)       : {(good_confidence-high_confidence):4d} ({(good_confidence-high_confidence)/total*100:5.1f}%)")
    print(f"  • 0.70-0.79 (Moderate)   : {(moderate_confidence-good_confidence):4d} ({(moderate_confidence-good_confidence)/total*100:5.1f}%)")
    print(f"  • 0.60-0.69 (Low)        : {(low_confidence-moderate_confidence):4d} ({(low_confidence-moderate_confidence)/total*100:5.1f}%)")
    print(f"  • <0.60 (Very Low)       : {very_low:4d} ({very_low/total*100:5.1f}%)")


Confidence Score (match1_score) Statistics:
  Mean score    : 0.8240
  Median score  : 0.8200
  Std dev       : 0.1024
  Min score     : 0.6500
  Max score     : 0.9600
  25th percentile: 0.7575
  75th percentile: 0.9075

Score Distribution by Confidence Level:
  • 0.90-1.00 (Excellent)  :    3 ( 30.0%)
  • 0.80-0.89 (Good)       :    3 ( 30.0%)
  • 0.70-0.79 (Moderate)   :    3 ( 30.0%)
  • 0.60-0.69 (Low)        :    1 ( 10.0%)
  • <0.60 (Very Low)       :    0 (  0.0%)


## 4. Calculate Accuracy Metrics

In [7]:
if df_mapper_output is not None and df_curated is not None:
    print("\n" + "="*80)
    print("ACCURACY METRICS CALCULATION")
    print("="*80)
    
    # Create a mapping dictionary from mapper output
    # Key: query (raw field), Value: match1 (mapped field)
    mapper_dict = dict(zip(df_mapper_output['query'], df_mapper_output['match1']))
    
    # Gold standard: NEW metadata columns should map to CURATED columns
    # For simplicity, we'll check if match1 is a valid standard field
    standard_fields = set(df_curated.columns)
    
    # Evaluate each mapping
    correct_mappings = 0
    partially_correct = 0  # If match1 contains the concept but different
    incorrect_mappings = 0
    high_confidence_correct = 0
    
    evaluation_results = []
    
    for idx, row in df_mapper_output.iterrows():
        query = row['query']
        match1 = row['match1']
        score = pd.to_numeric(row['match1_score'], errors='coerce')
        method = row.get('method', 'unknown')
        
        # Check if match1 is in standard fields
        if match1 in standard_fields:
            correct_mappings += 1
            if score >= 0.90:
                high_confidence_correct += 1
            result = "Valid"
        else:
            incorrect_mappings += 1
            result = "Invalid"
        
        evaluation_results.append({
            'query': query,
            'match1': match1,
            'score': score,
            'method': method,
            'valid': result
        })
    
    # Create evaluation dataframe
    evaluation_df = pd.DataFrame(evaluation_results)
    
    # Calculate metrics
    total_mappings = len(evaluation_df)
    accuracy = (correct_mappings / total_mappings) * 100 if total_mappings > 0 else 0
    high_conf_accuracy = (high_confidence_correct / correct_mappings) * 100 if correct_mappings > 0 else 0
    
    print(f"\nACCURACY METRICS:")
    print(f"  Total mappings evaluated    : {total_mappings}")
    print(f"  Correct mappings (match1 in standard): {correct_mappings} ({accuracy:.1f}%)")
    print(f"  Incorrect mappings                   : {incorrect_mappings} ({100-accuracy:.1f}%)")
    print(f"\n  High confidence correct (>=0.90)  : {high_confidence_correct}")
    print(f"  % of correct with high confidence: {high_conf_accuracy:.1f}%")


ACCURACY METRICS CALCULATION

ACCURACY METRICS:
  Total mappings evaluated    : 10
  Correct mappings (match1 in standard): 3 (30.0%)
  Incorrect mappings                   : 7 (70.0%)

  High confidence correct (>=0.90)  : 1
  % of correct with high confidence: 33.3%


## 5. Detailed Evaluation by Method

In [8]:
if df_mapper_output is not None:
    print("\n" + "="*80)
    print("EVALUATION BY MATCHING METHOD")
    print("="*80)
    
    if 'method' in df_mapper_output.columns:
        method_analysis = []
        
        for method in df_mapper_output['method'].unique():
            method_rows = df_mapper_output[df_mapper_output['method'] == method]
            count = len(method_rows)
            avg_score = pd.to_numeric(method_rows['match1_score'], errors='coerce').mean()
            
            # Count valid mappings for this method
            valid_count = (method_rows['match1'].isin(standard_fields)).sum()
            valid_rate = (valid_count / count) * 100 if count > 0 else 0
            
            method_analysis.append({
                'Method': method,
                'Count': count,
                'Avg_Score': f"{avg_score:.4f}",
                'Valid': valid_count,
                'Valid_%': f"{valid_rate:.1f}%"
            })
        
        method_df = pd.DataFrame(method_analysis)
        print("\n")
        print(method_df.to_string(index=False))


EVALUATION BY MATCHING METHOD


  Method  Count Avg_Score  Valid Valid_%
   exact      3    0.9433      1   33.3%
   fuzzy      3    0.8367      1   33.3%
semantic      3    0.7500      1   33.3%
     llm      1    0.6500      0    0.0%


## 6. Failure Case Analysis

In [9]:
if df_mapper_output is not None:
    print("\n" + "="*80)
    print("FAILURE CASE ANALYSIS")
    print("="*80)
    
    # Get invalid mappings
    invalid_mappings = df_mapper_output[~df_mapper_output['match1'].isin(standard_fields)]
    
    if len(invalid_mappings) > 0:
        print(f"\nFields that failed to map to standard vocabulary: {len(invalid_mappings)}\n")
        
        # Sort by confidence score
        invalid_mappings = invalid_mappings.sort_values('match1_score', ascending=False)
        
        print("Top 15 INVALID mappings (by score):")
        print()
        
        for idx, (i, row) in enumerate(invalid_mappings.head(15).iterrows(), 1):
            print(f"{idx:2d}. Query: {str(row['query']):30s} → {str(row['match1']):40s} (Score: {row['match1_score']:.4f})")
    else:
        print("\nAll mappings are valid (100% success rate)!")


FAILURE CASE ANALYSIS

Fields that failed to map to standard vocabulary: 7

Top 15 INVALID mappings (by score):

 1. Query: sample_col_8                   → biological_sex                           (Score: 0.9600)
 2. Query: column_4                       → disease_category                         (Score: 0.9200)
 3. Query: column_2                       → biological_sex                           (Score: 0.8700)
 4. Query: sample_col_9                   → specimen_location                        (Score: 0.8300)
 5. Query: column_3                       → specimen_location                        (Score: 0.7800)
 6. Query: sample_col_7                   → disease_category                         (Score: 0.7200)
 7. Query: column_5                       → body_mass_index                          (Score: 0.6500)


## 7. Low Confidence Mappings

In [10]:
if df_mapper_output is not None and 'match1_score' in df_mapper_output.columns:
    print("\n" + "="*80)
    print("LOW CONFIDENCE MAPPINGS (Score < 0.80)")
    print("="*80)
    
    df_mapper_output['match1_score_numeric'] = pd.to_numeric(df_mapper_output['match1_score'], errors='coerce')
    low_conf = df_mapper_output[df_mapper_output['match1_score_numeric'] < 0.80].sort_values('match1_score_numeric')
    
    if len(low_conf) > 0:
        print(f"\n{len(low_conf)} mappings with confidence < 0.80 (need manual review)\n")
        print("Top 20 low confidence mappings:\n")
        
        for idx, (i, row) in enumerate(low_conf.head(20).iterrows(), 1):
            print(f"{idx:2d}. {str(row['query']):25s} → {str(row['match1']):35s} (Score: {row['match1_score']:.4f})")
    else:
        print("\nAll mappings have confidence >= 0.80!")


LOW CONFIDENCE MAPPINGS (Score < 0.80)

4 mappings with confidence < 0.80 (need manual review)

Top 20 low confidence mappings:

 1. column_5                  → body_mass_index                     (Score: 0.6500)
 2. sample_col_7              → disease_category                    (Score: 0.7200)
 3. sample_col_10             → age_years                           (Score: 0.7500)
 4. column_3                  → specimen_location                   (Score: 0.7800)


## 8. Confidence Score Reliability

In [11]:
if df_mapper_output is not None and 'match1_score' in df_mapper_output.columns:
    print("\n" + "="*80)
    print("CONFIDENCE SCORE RELIABILITY")
    print("="*80)
    
    # Check if high scores correlate with valid mappings
    if 'evaluation_results' in dir():
        eval_df = pd.DataFrame(evaluation_results)
        
        # Correlation between score and correctness
        eval_df['is_valid'] = (eval_df['valid'] == 'Valid').astype(int)
        correlation = eval_df['score'].corr(eval_df['is_valid'])
        
        print(f"\nCorrelation between score and validity: {correlation:.4f}")
        print("   (1.0 = perfect correlation, 0.0 = no correlation)")
        
        # Precision at different score thresholds
        print(f"\nPrecision at Different Score Thresholds:")


CONFIDENCE SCORE RELIABILITY

Correlation between score and validity: 0.0853
   (1.0 = perfect correlation, 0.0 = no correlation)

Precision at Different Score Thresholds:


## 9. Recommendations

In [15]:
print("\n" + "="*80)
print("EVALUATION RECOMMENDATIONS")
print("="*80)

recommendations = """
BASED ON EVALUATION RESULTS:

1. ACCURACY ASSESSMENT
   • If accuracy > 95%: Mapper is performing excellently
   • If accuracy 80-95%: Good performance, some fields need review
   • If accuracy 60-80%: Moderate performance, needs tuning
   • If accuracy < 60%: Poor performance, needs major revision

2. CONFIDENCE SCORE USAGE
   • Score > 0.90: AUTO-ACCEPT these mappings (minimal review)
   • Score 0.80-0.90: REVIEW these mappings (curator decision)
   • Score 0.70-0.80: FLAG for detailed review
   • Score < 0.70: MANUAL mapping recommended

3. PROBLEM FIELDS
   • Identify fields with consistently low scores
   • Consider: Are field names ambiguous? Values inconsistent?
   • May need dictionary/ontology updates

4. METHOD PERFORMANCE
   • Which methods (exact, fuzzy, semantic, llm) perform best?
   • Can we improve by adjusting method thresholds?
   • Consider tuning transition between stages

5. COMMON FAILURE PATTERNS
   • Free text fields (disease, treatment descriptions)
   • Abbreviated/coded values (M vs Male, T2DM vs "diabetes mellitus type 2")
   • Field name variations (sex vs gender, age_years vs age)

6. CURATOR TOOL REQUIREMENTS
   • Show top 3 matches for every field (allow override)
   • Highlight low-confidence mappings (< 0.80)
   • Bulk accept high-confidence mappings (>= 0.90)
   • Show example values from raw data during review
   • Save curator decisions for future improvement

7. NEXT ITERATION IMPROVEMENTS
   • Retrain on curator feedback
   • Update ontology/value dictionaries
   • Adjust confidence thresholds
   • Add domain-specific rules for recurring patterns
"""

print(recommendations)


EVALUATION RECOMMENDATIONS

BASED ON EVALUATION RESULTS:

1. ACCURACY ASSESSMENT
   • If accuracy > 95%: Mapper is performing excellently
   • If accuracy 80-95%: Good performance, some fields need review
   • If accuracy 60-80%: Moderate performance, needs tuning
   • If accuracy < 60%: Poor performance, needs major revision

2. CONFIDENCE SCORE USAGE
   • Score > 0.90: AUTO-ACCEPT these mappings (minimal review)
   • Score 0.80-0.90: REVIEW these mappings (curator decision)
   • Score 0.70-0.80: FLAG for detailed review
   • Score < 0.70: MANUAL mapping recommended

3. PROBLEM FIELDS
   • Identify fields with consistently low scores
   • Consider: Are field names ambiguous? Values inconsistent?
   • May need dictionary/ontology updates

4. METHOD PERFORMANCE
   • Which methods (exact, fuzzy, semantic, llm) perform best?
   • Can we improve by adjusting method thresholds?
   • Consider tuning transition between stages

5. COMMON FAILURE PATTERNS
   • Free text fields (disease, treatm

## 10. Summary Report

In [13]:
print("\n" + "="*80)
print("EVALUATION SUMMARY")
print("="*80)

if df_mapper_output is not None:
    summary_report = f"""
SCHEMA MAPPER EVALUATION REPORT

DATASET SUMMARY
   • Raw input columns: {len(df_mapper_output)}
   • Standard output fields: {len(standard_fields)}
   • Mapping coverage: {len(df_mapper_output)} → {len(standard_fields)}

ACCURACY METRICS
   • Overall accuracy: {accuracy:.1f}%
   • Valid mappings: {correct_mappings}/{total_mappings}
   • Invalid mappings: {incorrect_mappings}/{total_mappings}

CONFIDENCE ANALYSIS
   • Average score: {df_mapper_output['match1_score_numeric'].mean():.4f}
   • Median score: {df_mapper_output['match1_score_numeric'].median():.4f}
   • Excellent (>=0.90): {(df_mapper_output['match1_score_numeric'] >= 0.90).sum()} mappings
   • Low (<0.70): {(df_mapper_output['match1_score_numeric'] < 0.70).sum()} mappings

QUALITY ASSESSMENT
   • Fields needing manual review: {len(low_conf)}
   • High confidence + valid mappings: {high_confidence_correct}
   • Ready for auto-acceptance (>=0.90): {(df_mapper_output['match1_score_numeric'] >= 0.90).sum()}

RECOMMENDATIONS
   1. Auto-accept {(df_mapper_output['match1_score_numeric'] >= 0.90).sum()} high-confidence mappings (>=0.90)
   2. Curator review {len(low_conf)} low-confidence mappings (<0.80)
   3. Use match2/match3 alternatives for ambiguous cases
   4. Collect curator feedback to improve mapper for next iteration
"""
    
    print(summary_report)


EVALUATION SUMMARY

SCHEMA MAPPER EVALUATION REPORT

DATASET SUMMARY
   • Raw input columns: 10
   • Standard output fields: 37
   • Mapping coverage: 10 → 37

ACCURACY METRICS
   • Overall accuracy: 30.0%
   • Valid mappings: 3/10
   • Invalid mappings: 7/10

CONFIDENCE ANALYSIS
   • Average score: 0.8240
   • Median score: 0.8200
   • Excellent (>=0.90): 3 mappings
   • Low (<0.70): 1 mappings

QUALITY ASSESSMENT
   • Fields needing manual review: 4
   • High confidence + valid mappings: 1
   • Ready for auto-acceptance (>=0.90): 3

RECOMMENDATIONS
   1. Auto-accept 3 high-confidence mappings (>=0.90)
   2. Curator review 4 low-confidence mappings (<0.80)
   3. Use match2/match3 alternatives for ambiguous cases
   4. Collect curator feedback to improve mapper for next iteration



## 11. Export Results

In [14]:
# Save evaluation results for reference
if 'evaluation_df' in dir():
    # Save full evaluation
    evaluation_df.to_csv('mapper_evaluation_results.csv', index=False)
    print("Saved: mapper_evaluation_results.csv")
    
    # Save summary metrics
    if 'method_df' in dir():
        method_df.to_csv('method_performance_summary.csv', index=False)
        print("Saved: method_performance_summary.csv")
    
    # Save invalid mappings for manual review
    if len(invalid_mappings) > 0:
        invalid_mappings[['query', 'match1', 'match1_score', 'method']].to_csv('invalid_mappings_for_review.csv', index=False)
        print("Saved: invalid_mappings_for_review.csv")
    
    print("\nAll evaluation results exported to analysis/ directory")

Saved: mapper_evaluation_results.csv
Saved: method_performance_summary.csv
Saved: invalid_mappings_for_review.csv

All evaluation results exported to analysis/ directory
